In [17]:
"""
Qwen-Rerank LoRA微调，CrossEncoder二分类
数据集 rerank_dataset.csv：text,label
MODEL_NAME = "Qwen/Qwen2-Rerank-0.5B"
"""
import torch
import pandas as pd
from datasets import Dataset
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    TrainingArguments,
    Trainer,
)
from peft import LoraConfig, get_peft_model
from evaluate import load

In [18]:
# ===================== 配置 =====================
MODEL_NAME = "Qwen/Qwen3-Reranker-0.6B"
MAX_SEQ_LEN = 512
BATCH_SIZE = 8
EPOCHS = 3
LR = 2e-4
OUTPUT_DIR = "./qwen-rerank-lora"

# LoRA配置，Qwen是Decoder-only，target_modules参考Qwen2
lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=[
        "q_proj",
        "v_proj",
    ],
    lora_dropout=0.05,
    bias="none",
    task_type="SEQ_CLS"
)

In [19]:
# ===================== 加载数据集 =====================
df = pd.read_csv("rerank_dataset.csv")
raw_ds = Dataset.from_pandas(df)
split_ds = raw_ds.train_test_split(test_size=0.1, seed=42)
train_ds = split_ds["train"]
val_ds = split_ds["test"]

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
# 1. tokenizer层面设置pad
tokenizer.pad_token = tokenizer.eos_token
tokenizer.pad_token_id = tokenizer.eos_token_id

def tokenize_fn(examples):
    return tokenizer(
        examples["text"],
        truncation=True,
        padding="max_length",
        max_length=MAX_SEQ_LEN
    )

train_ds = train_ds.map(tokenize_fn, batched=True)
val_ds = val_ds.map(tokenize_fn, batched=True)

train_ds.set_format("torch", columns=["input_ids", "attention_mask", "label"])
val_ds.set_format("torch", columns=["input_ids", "attention_mask", "label"])

Map:   0%|          | 0/4 [00:00<?, ? examples/s]

Map:   0%|          | 0/1 [00:00<?, ? examples/s]

In [25]:
# ===================== 加载模型 =====================
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2,
    torch_dtype=torch.bfloat16 if torch.cuda.is_available() else torch.float32,
    trust_remote_code=True
)
# 2. 【关键】同步更新模型config的pad_token_id，解决报错
model.config.pad_token_id = tokenizer.eos_token_id
model.config.eos_token_id = tokenizer.eos_token_id

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

Loading weights:   0%|          | 0/310 [00:00<?, ?it/s]

[transformers] Qwen3ForSequenceClassification LOAD REPORT from: Qwen/Qwen3-Reranker-0.6B
Key          | Status  | 
-------------+---------+-
score.weight | MISSING | 

Notes:
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


trainable params: 1,148,928 || all params: 596,927,488 || trainable%: 0.1925


In [26]:
# ===================== 评估指标 =====================
metric = load("accuracy")
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = logits.argmax(axis=-1)
    return metric.compute(predictions=preds, references=labels)

In [27]:
# ===================== 训练参数 =====================
training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    learning_rate=LR,
    logging_steps=10,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    bf16=torch.cuda.is_available(),
    fp16=False,
    report_to="none",
    save_total_limit=2
)

In [28]:
# ===================== Trainer =====================
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    compute_metrics=compute_metrics
)

if __name__ == "__main__":
    trainer.train()
    trainer.save_model(OUTPUT_DIR)
    tokenizer.save_pretrained(OUTPUT_DIR)
    print(f"Qwen3-Reranker LoRA保存至：{OUTPUT_DIR}")

Epoch,Training Loss,Validation Loss,Accuracy
1,No log,1.328125,0.000000
2,No log,1.593750,0.000000
3,No log,1.445312,0.000000


[transformers] mtime may not be reliable on this filesystem, falling back to numerical ordering


Qwen3-Reranker LoRA保存至：./qwen-rerank-lora
